In [ ]:
# Install libraries and define working directory - Make sure to adjust the working directory.
import sys
import os
!{sys.executable} -m pip install --upgrade google-cloud-bigquery google-cloud-storage pycountry pandas-gbq pyarrow
os.chdir('/your/working/directory') # # Adjust to your working directory, so the whole project can be run from the root folder and the paths to data and database are consistent across functions
print("LIBRARIES INSTALLED AND WD DEFINED IN")
print(os.getcwd())


In [ ]:
# Libraries
import time
import warnings
import pycountry
import pandas as pd
import pyarrow
from google.cloud import bigquery

# Environment configuration - Make sure to replace 'YOUR-GOOGLE-BIGQUERY-CREDENTIALS.json' with the actual name of your credentials file.
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "data/YOUR-GOOGLE-BIGQUERY-CREDENTIALS.json" # Remember: 'data' folder should be in the root of the project, and the credentials file to retrieve data from Google's platform should be inside it. 
warnings.filterwarnings("ignore")

# Directory setup
output_dir = "chips_2026/data/Google-patents" # This is where the output of this notebook will be stored.

# BigQuery Client Initialization
PROJECT_ID = 'patents-chips'
DATASET_ID = 'data'
client = bigquery.Client(project=PROJECT_ID)

# Country configuration and ISO2 conversion
foci = [ # Selection is based on literature review for the case of interest, but you can modify according to your needs.
    "USA", "Austria", "Belgium", "Bulgaria", "Croatia", "Cyprus",
    "Czechia", "Denmark", "Estonia", "Finland", "France",
    "Germany", "Greece", "Hungary", "Ireland", "Italy",
    "Latvia", "Lithuania", "Luxembourg", "Malta", "Netherlands",
    "China", "Taiwan", "Japan", "South Korea"
]

def get_iso2(name):
    try:
        if name == "USA": return "US" # pycountry uses 'US' for the United States, so we need to handle this case separately
        return pycountry.countries.search_fuzzy(name)[0].alpha_2
    except:
        return None

iso2_codes = [get_iso2(country) for country in foci if get_iso2(country) is not None]
iso2_string = ", ".join(f"'{c}'" for c in set(iso2_codes + ["WO", "EP"]))

# SQL Query definition. This is where we extract the relevant information from the BigQuery patents database, applying the necessary transformations to create the features we need for our analysis.
query = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.raw` AS
SELECT 
  publication_number AS patent_id,
  country_code, 
  
  -- 1. Extract the specific H10 code found in the array
  (SELECT c.code FROM UNNEST(cpc) AS c 
   WHERE STARTS_WITH(c.code, 'H10') 
   LIMIT 1) AS specific_h_code,

  -- 2. Classification Signal logic
  CASE 
    WHEN STARTS_WITH(cpc[SAFE_OFFSET(0)].code, 'H10') THEN 'H10'
    ELSE 'OTHER FIELD'
  END AS main_field_type,

  -- 3. Extract the Class (first 3 chars) if it is not H04 or H10
  CASE
    WHEN STARTS_WITH(cpc[SAFE_OFFSET(0)].code, 'H10') 
    THEN NULL
    ELSE SUBSTR(cpc[SAFE_OFFSET(0)].code, 1, 3)
  END AS which_other_main_field,

  DIV(publication_date, 10000) AS pub_year, 
  (family_id IS NOT NULL AND family_id != '-1') AS is_global_patent,
  family_id,

  -- 4. Flattened strings for inventors and assignees
    (SELECT STRING_AGG(inv.name, " - ") FROM UNNEST(inventor_harmonized) AS inv) AS inventor_names,
  (SELECT STRING_AGG(ass.name, " - ") FROM UNNEST(assignee_harmonized) AS ass) AS assignee_names,

  -- 5. Same for countries of inventors and assignees
  (SELECT STRING_AGG(inv.country_code, " - ") FROM UNNEST(inventor_harmonized) AS inv) AS inventor_countries,
  (SELECT STRING_AGG(ass.country_code, " - ") FROM UNNEST(assignee_harmonized) AS ass) AS assignee_countries

FROM 
  `patents-public-data.patents.publications`
WHERE 
  EXISTS(
    SELECT 1 FROM UNNEST(cpc) AS c 
    WHERE STARTS_WITH(c.code, 'H10')
  )
  AND country_code IN ({iso2_string})
  AND publication_date BETWEEN 19800101 AND 20261231
"""
# 1. Query execution
print(f"--- Starting process for {len(iso2_codes)} jurisdictions ---")
start_query = time.time()
client.query(query).result() 

print(f"Success: Table `{PROJECT_ID}.{DATASET_ID}.raw` created/updated.")
print(f"Processing time: {time.time() - start_query:.2f} seconds.\n")

# 2. Fetching the 50-row sample, just to be sure the information needed is included.
print("Fetching 50-row sample...")
# We query the newly created table with a LIMIT to save costs/memory
query_sample = f"SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.raw` LIMIT 50"

# Execute and convert to dataframe
df_sample = client.query(query_sample).to_dataframe(create_bqstorage_client=False)

# 3. Display results
print("-" * 30)
print("DATA SAMPLE (First 50 rows):")
print("-" * 30)

# Configure pandas to show all columns in the console output
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Print the sample
print(df_sample)

print(f"\nSample dimensions: {df_sample.shape}")

In [ ]:
# Now query for summarized data
query_summary = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.summary` AS
SELECT 
  pub_year,
  country_code,

  -- (0) Total unique patents for each country-year combination
  COUNT(DISTINCT patent_id) AS total_unique_patents,
  
  -- (1) Total number of H10 as main field
  COUNT(DISTINCT CASE WHEN main_field_type = 'H10' THEN patent_id END) AS total_mainfield_h10,
  
  -- (2) Repeat counting only 'global patents' (Unique IDs)
  COUNT(DISTINCT CASE WHEN main_field_type = 'H10' AND is_global_patent THEN patent_id END) AS global_h10,

  -- (3) Patents where country_code != (assignee OR inventor country)
  -- Uses DISTINCT to ensure unique patent counts
  COUNT(DISTINCT CASE 
    WHEN NOT REGEXP_CONTAINS(assignee_countries, country_code) OR 
         NOT REGEXP_CONTAINS(inventor_countries, country_code) 
    THEN patent_id END) AS foreign_owned_patents

FROM 
  `{PROJECT_ID}.{DATASET_ID}.raw`
GROUP BY 
  pub_year, 
  country_code
ORDER BY 
  pub_year DESC, 
  total_unique_patents DESC
"""

# --- EXECUTE SUMMARY TABLE CREATION ---
print("Creating Summary table...")
client.query(query_summary).result()
print(f"Success: Table `{PROJECT_ID}.{DATASET_ID}.summary` created.")

# --- FETCH AND DISPLAY SAMPLE ---
print("Fetching summary sample...")
query_fetch_summary = f"SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.summary` LIMIT 50"
rows = client.query(query_fetch_summary).result()

# Convert to DataFrame manually to avoid library version conflicts
df_summary = pd.DataFrame([dict(row) for row in rows])

# Display formatting
print("-" * 30)
print("SUMMARY TABLE SAMPLE (First 50 groups):")
print("-" * 30)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
print(df_summary)

In [ ]:
# 1. Configuration
output_dir = "data/Google-patents" # This is where the output of this notebook will be stored. Repetition is just to ensure the variable is defined in this cell.
output_file = "patents_summary_2026.parquet" # Name of the output file, which will be stored in parquet format for efficient storage and later use in the analysis notebook.
output_path = os.path.join(output_dir, output_file)

# Ensure the directory exists
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 2. Fetch all data from the summary table
print(f"Downloading data from {PROJECT_ID}.{DATASET_ID}.summary...")
sql_fetch = f"SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.summary`"
query_job = client.query(sql_fetch)
rows = query_job.result()

# 3. Convert to DataFrame (Safe manual method)
df_final = pd.DataFrame([dict(row) for row in rows])

# 4. Export to Parquet
if not df_final.empty:
    print(f"Exporting {len(df_final)} rows to Parquet...")
    # engine='pyarrow' is used here to ensure data types are preserved
    df_final.to_parquet(output_path, engine='pyarrow', compression='snappy', index=False)
    print(f"File successfully saved to: {output_path}")
else:
    print("Warning: The summary table is empty. No file was created.")